# MeMo Torch 
Version integrated with Torch (Keras)

In [1]:
import torch

from MeMoPyTorch.modelling_memo import MeMo
from MeMoPyTorch.modelling_memo_tokenizer import MeMoTokenizer
from MeMoPyTorch.evaluating_memo import Evaluation

from MeMoHF.utils import seed_everything
seed_everything(42)

/opt/dev/anaconda3/envs/deepai/lib/python3.11/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Memo: Initializing the Tokenizer and the model

In [2]:
# Meta Parameters : 
#    d - inner dimension
#    h - number of heads
#    l - number of layers
d,h,l = 2048, 4, 3
chunk_length = 4096

# Initializing a standard Tokenizer
max_length = chunk_length 
tokenizer = MeMoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b", 
                                          padding_side='left', truncation_side='left', 
                                          max_length=max_length, head_number=h)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.pad_token_id

device = 'cpu'
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
    device = 'cuda'

# Intializing Memo 
model = MeMo(inner_dim=d, 
             num_of_heads=h, 
             num_of_layers=l, 
             chunk_length=max_length, 
             num_embeddings=tokenizer.vocab_size, 
             padding_idx=tokenizer.pad_token_id, 
             device=device)


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPTNeoXTokenizer'. 
The class this function is called from is 'MeMoTokenizer'.


Setting pad token and pad token id = <|endoftext|>, 0
GPU: NVIDIA GeForce RTX 3080 Ti Laptop GPU is available.
MeMo embedding initilialization


Reading the two texts

In [3]:
with open("testo_di_prova.txt") as my_first_text_f:
    my_first_text = ''.join(my_first_text_f.read().split()[:50])
with open("testo_di_prova2.txt") as my_first_text_f:
    my_second_text = my_first_text_f.read()



Memorizing the first text and evaluating if it is memorized

In [4]:
memo_input_1 = tokenizer.get_text_batch_encoding([my_first_text]*1)  # Writing the same doc 8 times to stress the memorization with batch
memo_input_2 = tokenizer.get_text_batch_encoding([my_second_text]*1) # Writing the same doc 8 times to stress the memorization with batch

model.memorize_text(memo_input_1)
e = Evaluation()

e1 = e.check_pretokenized(model, tokenizer, memo_input_1['input_ids'], starting_point=0)# starting_point=8)
e2 = e.check_pretokenized(model, tokenizer, memo_input_2['input_ids'], starting_point=0)# starting_point=8)

print("Memorization level of first text  : ", e1) 
print("Memorization level of second text : ", e2) 

Starting point : 0


100%|██████████| 4095/4095 [00:06<00:00, 603.58it/s]


Starting point : 0


100%|██████████| 4095/4095 [00:06<00:00, 659.46it/s]

Memorization level of first text  :  tensor(0.9896)
Memorization level of second text :  tensor(0.0040)


Memorizing the second text and checking if it affected the memorization of the first text

In [5]:
model.memorize_text(memo_input_2)

e1 = e.check_pretokenized(model, tokenizer, memo_input_1['input_ids'], starting_point=0)# starting_point=8)
e2 = e.check_pretokenized(model, tokenizer, memo_input_2['input_ids'], starting_point=0)# starting_point=8)

print("Memorization level of first text  : ", e1) 
print("Memorization level of second text : ", e2) 

Starting point : 0


100%|██████████| 4095/4095 [00:06<00:00, 645.56it/s]


Starting point : 0


100%|██████████| 4095/4095 [00:06<00:00, 670.20it/s]

Memorization level of first text  :  tensor(0.9896)
Memorization level of second text :  tensor(0.9968)


Forgetting the first document

In [6]:
model.forget_text(memo_input_2)

Checking the effect on the two texts

In [7]:
e1 = e.check_pretokenized(model, tokenizer, memo_input_1['input_ids'], starting_point=0)# starting_point=8)
e2 = e.check_pretokenized(model, tokenizer, memo_input_2['input_ids'], starting_point=0)# starting_point=8)

print("Memorization level of first text  : ", e1) 
print("Memorization level of second text : ", e2) 

Starting point : 0


100%|██████████| 4095/4095 [00:06<00:00, 667.76it/s]


Starting point : 0


100%|██████████| 4095/4095 [00:06<00:00, 672.90it/s]

Memorization level of first text  :  tensor(0.9896)
Memorization level of second text :  tensor(0.0040)


In [ ]:
exit()

: 